# Unwrapping AI — Step 06: Regularization
Regularization helps control model complexity by penalising large coefficients.

**Learning objectives**
- Understand why regularization is useful.
- Compare Linear Regression, Ridge (L2) and Lasso (L1).
- Observe how the regularization strength changes coefficients and test performance.
- Understand why scaling matters for regularized models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

## 1. Load the dataset and split the data

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Features:", list(X.columns))

## 2. Train three models

- **Linear Regression:** no coefficient penalty.
- **Ridge:** L2 penalty; discourages large coefficients.
- **Lasso:** L1 penalty; can shrink some coefficients to zero.

We scale features inside each pipeline so that regularization treats features fairly.

In [ ]:
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (alpha=1)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (alpha=0.01)": make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=20000))
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Test MSE": mean_squared_error(y_test, pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "Test R2": r2_score(y_test, pred)
    })

pd.DataFrame(results).round(4)

## 3. Compare Ridge regularization strengths

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_results = []

for alpha in alphas:
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    ridge_results.append({
        "Alpha": alpha,
        "Test MSE": mean_squared_error(y_test, pred),
        "Test R2": r2_score(y_test, pred)
    })

ridge_df = pd.DataFrame(ridge_results)
display(ridge_df.round(4))

plt.plot(ridge_df["Alpha"], ridge_df["Test MSE"], marker="o")
plt.xscale("log")
plt.xlabel("Ridge alpha (log scale)")
plt.ylabel("Test MSE")
plt.title("Effect of Ridge regularization strength")
plt.show()

## 4. Compare coefficient sizes

In [ ]:
alpha_values = [0.01, 1, 100]
coefficient_table = pd.DataFrame(index=X.columns)

for alpha in alpha_values:
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    coefficient_table[f"Ridge alpha={alpha}"] = model.named_steps["ridge"].coef_

display(coefficient_table.round(4))

coefficient_table.plot(kind="bar", figsize=(12, 5))
plt.ylabel("Coefficient value")
plt.title("How Ridge regularization changes coefficients")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Observe Lasso feature selection

In [ ]:
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.05, max_iter=20000))
lasso.fit(X_train, y_train)
lasso_coefs = lasso.named_steps["lasso"].coef_

lasso_summary = pd.DataFrame({
    "Feature": X.columns,
    "Lasso coefficient": lasso_coefs,
    "Coefficient is zero": np.isclose(lasso_coefs, 0)
})
display(lasso_summary.round(4))
print("Number of zero coefficients:", np.isclose(lasso_coefs, 0).sum())

## 6. Key concepts

| Method | Penalty | Typical effect |
|---|---|---|
| Linear Regression | None | Fits without coefficient penalty |
| Ridge | L2 (squared coefficients) | Shrinks coefficients towards zero |
| Lasso | L1 (absolute coefficients) | Can set coefficients exactly to zero |

**Alpha** controls penalty strength. A very small alpha has little effect; an excessively large alpha can cause underfitting. Choose it using validation or cross-validation, not by repeatedly tuning against the final test set.

## Knowledge check
1. Why can regularization reduce overfitting?
2. What is the difference between L1 and L2?
3. Why should features usually be scaled before Ridge or Lasso?
4. What happens when alpha becomes extremely large?
5. Why should we use cross-validation to select alpha?

## Completion checklist
- [ ] Run all cells in Google Colab.
- [ ] Compare the three models.
- [ ] Inspect the Ridge alpha chart.
- [ ] Observe coefficient shrinkage and Lasso zeros.
- [ ] Save the executed notebook and upload it to GitHub.

**Takeaway:** Regularization discourages unnecessarily complex models. It can improve generalisation, but the penalty strength must be chosen carefully.